# Extracción de Datos (Web Scraping)

Este notebook se encarga de extraer, cargar y consolidar los datos crudos de diferentes temporadas de LaLiga.

## 1. Importación de Librerías y Configuración
En esta primera celda, importamos las herramientas necesarias (`pandas`, `requests`, `BeautifulSoup`, etc.) y definimos las rutas de los directorios de trabajo y las columnas requeridas para nuestro análisis.


In [1]:
from pathlib import Path
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup


BASE_DIR = Path.cwd().parent if Path.cwd().name == "web_scraping" else Path.cwd()
RAW_DIR = BASE_DIR / "web_scraping" / "raw"
PROCESSED_DIR = BASE_DIR / "web_scraping" / "processed"
REQUIRED_COLUMNS = [
    "player_name",
    "team",
    "position",
    "minutes_played",
    "goals",
    "assists",
    "yellow_cards",
    "red_cards",
]


## 2. Carga de Datos Crudos Locales
Definimos la función `load_raw_data`, la cual recorre el directorio de archivos crudos, lee los CSV de cada temporada filtrando solo las columnas de interés y añade una columna que identifica la temporada correspondiente. Finalmente, une todo en un único DataFrame.


In [2]:
def load_raw_data(raw_dir=RAW_DIR):
    dataframes = []
    for csv_path in sorted(raw_dir.glob("*.csv")):
        frame = pd.read_csv(csv_path, usecols=REQUIRED_COLUMNS)
        frame["season"] = csv_path.stem.split("-")[0].lower()
        dataframes.append(frame)
    if len(dataframes) != 4:
        raise FileNotFoundError("Se esperaban cuatro archivos CSV en web_scraping/raw")
    combined = pd.concat(dataframes, ignore_index=True)
    missing = set(REQUIRED_COLUMNS) - set(combined.columns)
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {sorted(missing)}")
    return combined


## 3. Web Scraping de Tablas de Jugadores
La función `scrape_player_table` permite realizar peticiones HTTP a una URL específica, procesar el HTML usando BeautifulSoup y extraer las filas de las tablas estadísticas, devolviendo un DataFrame con la información.


In [3]:
def scrape_player_table(url):
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    rows = []
    for row in soup.select("table tr"):
        values = [cell.get_text(" ", strip=True) for cell in row.select("th, td")]
        if len(values) == len(REQUIRED_COLUMNS):
            rows.append(dict(zip(REQUIRED_COLUMNS, values)))
    return pd.DataFrame(rows, columns=REQUIRED_COLUMNS)



## 4. Ejecución y Guardado de Datos
Finalmente, ejecutamos la función de carga para combinar la información de las temporadas, creamos el directorio de destino si no existe, y guardamos el conjunto de datos combinado en un nuevo archivo CSV listo para la fase de limpieza.


In [4]:
combined_data = load_raw_data()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
combined_data.to_csv(PROCESSED_DIR / "datos_combinados.csv", index=False)
combined_data.head()


,player_name,team,position,assists,goals,minutes_played,red_cards,yellow_cards,season
0,Aarón Escandell,Granada CF,Goalkeeper,NaN,NaN,283.0,1.0,1.0,s2122
1,Abdelkabir Abqar,Deportivo Alavés,Defender,NaN,NaN,NaN,NaN,NaN,s2122
2,Abdessamad Ezzalzouli,FC Barcelona,Forward,NaN,1.0,581.0,NaN,4.0,s2122
3,Abdón Prats,RCD Mallorca,Forward,NaN,3.0,633.0,NaN,4.0,s2122
4,Adama Traoré,FC Barcelona,Forward,2.0,NaN,378.0,NaN,NaN,s2122
